# ABCD Ray Transfer Matrix from `jax.jacobian` (Single Ray)

This notebook shows, for a **single paraxial ray**, that taking `jax.jacobian` of `run_to_end`
and then converting with `custom_jacobian_matrix` returns the expected **5x5 homogeneous ray
transfer matrix** for a model containing:

1. free-space propagation
2. a thin lens
3. free-space propagation
4. a deflector

State vector convention:

$\mathbf{r} = [x,\; y,\; \theta_x,\; \theta_y,\; 1]^T$


In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from temgym_core.ray import Ray
from temgym_core.components import Lens, Deflector
from temgym_core.run import run_to_end, solve_model
from temgym_core.utils import custom_jacobian_matrix

jax.config.update('jax_enable_x64', True)


## Model setup

We place the input ray at `z=0`, then a lens and a deflector at different `z` positions.
`run_to_end` inserts free-space propagators between these components automatically.


In [2]:
# Geometry and component parameters
z0 = 0.0
z_lens = 0.30
z_deflector = 0.80
f = 0.50
def_x = 1.2e-3
def_y = -0.8e-3

d1 = z_lens - z0
d2 = z_deflector - z_lens

ray_in = Ray(x=1.5e-3, y=-2.0e-3, dx=4.0e-3, dy=-3.0e-3, z=z0, pathlength=0.0)
model = (
    Lens(z=z_lens, focal_length=f),
    Deflector(z=z_deflector, def_x=def_x, def_y=def_y),
)

ray_out = run_to_end(ray_in, model)
ray_out


Ray(x=0.002, y=-0.0015, dx=-0.0002000000000000003, dy=0.0019999999999999996, z=0.8, pathlength=0.7999773, _one=1.0)

## Analytic 5x5 block matrices

For $\mathbf{r}=[x,y,\theta_x,\theta_y,1]^T$:

Free space over distance $d$:

$$
P(d)=\begin{bmatrix}
1&0&d&0&0\\
0&1&0&d&0\\
0&0&1&0&0\\
0&0&0&1&0\\
0&0&0&0&1
\end{bmatrix}
$$

Thin lens (focal length $f$):

$$
L(f)=\begin{bmatrix}
1&0&0&0&0\\
0&1&0&0&0\\
-1/f&0&1&0&0\\
0&-1/f&0&1&0\\
0&0&0&0&1
\end{bmatrix}
$$

Deflector (constant slope kick):

$$
D(\delta_x,\delta_y)=\begin{bmatrix}
1&0&0&0&0\\
0&1&0&0&0\\
0&0&1&0&\delta_x\\
0&0&0&1&\delta_y\\
0&0&0&0&1
\end{bmatrix}
$$

With our model ordering, expected total matrix is:

$$
M_{\mathrm{expected}} = D(\delta_x,\delta_y)\;P(d_2)\;L(f)\;P(d_1)
$$


In [3]:
def free_space_5x5(d):
    return jnp.array([
        [1.0, 0.0, d,   0.0, 0.0],
        [0.0, 1.0, 0.0, d,   0.0],
        [0.0, 0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 1.0],
    ], dtype=jnp.float64)


def lens_5x5(focal_length):
    return jnp.array([
        [1.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0, 0.0],
        [-1.0 / focal_length, 0.0, 1.0, 0.0, 0.0],
        [0.0, -1.0 / focal_length, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 1.0],
    ], dtype=jnp.float64)


def deflector_5x5(delta_x, delta_y):
    return jnp.array([
        [1.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0, delta_x],
        [0.0, 0.0, 0.0, 1.0, delta_y],
        [0.0, 0.0, 0.0, 0.0, 1.0],
    ], dtype=jnp.float64)

P1 = free_space_5x5(d1)
L = lens_5x5(f)
P2 = free_space_5x5(d2)
D = deflector_5x5(def_x, def_y)

M_expected = D @ P2 @ L @ P1
M_expected


Array([[ 0.0e+00,  0.0e+00,  5.0e-01,  0.0e+00,  0.0e+00],
       [ 0.0e+00,  0.0e+00,  0.0e+00,  5.0e-01,  0.0e+00],
       [-2.0e+00,  0.0e+00,  4.0e-01,  0.0e+00,  1.2e-03],
       [ 0.0e+00, -2.0e+00,  0.0e+00,  4.0e-01, -8.0e-04],
       [ 0.0e+00,  0.0e+00,  0.0e+00,  0.0e+00,  1.0e+00]], dtype=float64)

## Jacobian-derived 5x5 from TemGym

`jax.jacobian` of the ray mapping gives a Ray-shaped pytree of derivatives.
`custom_jacobian_matrix` converts that pytree to the standard 5x5 matrix ordering.


In [4]:
model_fn = lambda r: run_to_end(r, model)
ray_jac = jax.jacobian(model_fn)(ray_in)
M_jacobian = custom_jacobian_matrix(ray_jac)
M_jacobian


Array([[ 0.0e+00,  0.0e+00,  5.0e-01,  0.0e+00,  0.0e+00],
       [ 0.0e+00,  0.0e+00,  0.0e+00,  5.0e-01,  0.0e+00],
       [-2.0e+00,  0.0e+00,  4.0e-01,  0.0e+00,  1.2e-03],
       [ 0.0e+00, -2.0e+00,  0.0e+00,  4.0e-01, -8.0e-04],
       [ 0.0e+00,  0.0e+00,  0.0e+00,  0.0e+00,  1.0e+00]],      dtype=float64, weak_type=True)

In [5]:
np.testing.assert_allclose(np.asarray(M_jacobian), np.asarray(M_expected), rtol=1e-12, atol=1e-12)
print('Jacobian-derived 5x5 matches analytic ABCD matrix.')


Jacobian-derived 5x5 matches analytic ABCD matrix.


## Extra check: matrix action matches propagated output

Apply the 5x5 matrix to the input homogeneous ray vector and compare with `run_to_end`.


In [6]:
r_in_vec = jnp.array([ray_in.x, ray_in.y, ray_in.dx, ray_in.dy, ray_in._one], dtype=jnp.float64)
r_out_from_matrix = M_jacobian @ r_in_vec
r_out_direct = jnp.array([ray_out.x, ray_out.y, ray_out.dx, ray_out.dy, ray_out._one], dtype=jnp.float64)

np.testing.assert_allclose(np.asarray(r_out_from_matrix), np.asarray(r_out_direct), rtol=1e-12, atol=1e-12)
r_out_from_matrix, r_out_direct


(Array([ 2.0e-03, -1.5e-03, -2.0e-04,  2.0e-03,  1.0e+00], dtype=float64),
 Array([ 2.0e-03, -1.5e-03, -2.0e-04,  2.0e-03,  1.0e+00], dtype=float64))

## Per-step Jacobians (optional)

`solve_model` returns the Jacobian matrix at each propagation/component step.
Their ordered product reproduces the full system matrix.


In [7]:
step_mats = solve_model(ray_in, model)
M_from_steps = jnp.eye(5, dtype=jnp.float64)
for A in step_mats:
    M_from_steps = A @ M_from_steps

np.testing.assert_allclose(np.asarray(M_from_steps), np.asarray(M_jacobian), rtol=1e-12, atol=1e-12)
print('Per-step product equals full Jacobian 5x5 matrix.')
M_from_steps


Per-step product equals full Jacobian 5x5 matrix.


Array([[ 0.0e+00,  0.0e+00,  5.0e-01,  0.0e+00,  0.0e+00],
       [ 0.0e+00,  0.0e+00,  0.0e+00,  5.0e-01,  0.0e+00],
       [-2.0e+00,  0.0e+00,  4.0e-01,  0.0e+00,  1.2e-03],
       [ 0.0e+00, -2.0e+00,  0.0e+00,  4.0e-01, -8.0e-04],
       [ 0.0e+00,  0.0e+00,  0.0e+00,  0.0e+00,  1.0e+00]], dtype=float64)